# Comparação numérica: Base A (Crédito) x Base B (E-commerce)

O problema do Desafio 2 pede para escolher **uma** das duas bases oferecidas:

> "Você deverá atuar como um Cientista de Dados corporativo e construir um Pipeline
> Preditivo completo. Para isso, escolha UMA das duas bases de dados abaixo para
> resolver um problema real de mercado."

Este notebook existe só para isso: provar, com código executado (não com uma tabela
digitada à mão), os números usados para justificar a escolha da **base de crédito**
(Opção A) no lugar da base de e-commerce (Opção B). A conclusão e os números aqui
batem com a seção "Por que a base de crédito" do `README.md`, que é gerado a partir
do pipeline principal — este notebook é a evidência reprodutível por trás dela.

Nenhuma célula abaixo altera os dados originais nem os artefatos do pipeline
principal (`notebooks/03_executar_pipeline.py`); ele só lê os dois CSVs brutos.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

credito = pd.read_csv(ROOT / 'credit_risk_dataset.csv')
ecommerce = pd.read_csv(ROOT / 'E Commerce Dataset - E Comm.csv')

print('Crédito:   ', credito.shape[0], 'linhas x', credito.shape[1], 'colunas')
print('E-commerce:', ecommerce.shape[0], 'linhas x', ecommerce.shape[1], 'colunas')

Crédito:    32581 linhas x 12 colunas
E-commerce: 5630 linhas x 20 colunas


## 1. Tamanho e alvo

O alvo da base de crédito é `loan_status` (1 = inadimplente). O alvo da base de
e-commerce é `Churn` (1 = cliente que abandonou a plataforma). As duas são
binárias e desbalanceadas — mas em proporções diferentes.

In [2]:
alvo_credito = credito['loan_status'].value_counts(normalize=True).sort_index() * 100
alvo_ecommerce = ecommerce['Churn'].value_counts(normalize=True).sort_index() * 100

print('Crédito — loan_status (%):')
print(alvo_credito.round(2))
print()
print('E-commerce — Churn (%):')
print(alvo_ecommerce.round(2))

Crédito — loan_status (%):
loan_status
0    78.18
1    21.82
Name: proportion, dtype: float64

E-commerce — Churn (%):
Churn
0    83.16
1    16.84
Name: proportion, dtype: float64


## 2. Qualidade dos dados: nulos e duplicatas

Colunas com valores ausentes e linhas duplicadas são o tipo de problema que dá
material concreto para justificar decisões nas Etapas 1 e 2 (imputação, remoção de
duplicatas). Comparamos as duas bases pelo mesmo critério.

In [3]:
nulos_credito = credito.isnull().sum()
nulos_credito = nulos_credito[nulos_credito > 0]

nulos_ecommerce = ecommerce.isnull().sum()
nulos_ecommerce = nulos_ecommerce[nulos_ecommerce > 0]

print('Crédito — colunas com nulos:', len(nulos_credito))
print(nulos_credito)
print()
print('E-commerce — colunas com nulos:', len(nulos_ecommerce))
print(nulos_ecommerce)

Crédito — colunas com nulos: 2
person_emp_length     895
loan_int_rate        3116
dtype: int64

E-commerce — colunas com nulos: 7
Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64


In [4]:
dup_credito = credito.duplicated().sum()
# CustomerID torna cada linha única por definição; comparamos ignorando-o também,
# para não mascarar duplicatas reais de comportamento.
dup_ecommerce_com_id = ecommerce.duplicated().sum()
dup_ecommerce_sem_id = ecommerce.drop(columns=['CustomerID']).duplicated().sum()

print('Crédito — linhas inteiramente duplicadas:', dup_credito)
print('E-commerce — duplicadas (com CustomerID):', dup_ecommerce_com_id)
print('E-commerce — duplicadas (ignorando CustomerID):', dup_ecommerce_sem_id)

Crédito — linhas inteiramente duplicadas: 165
E-commerce — duplicadas (com CustomerID): 0
E-commerce — duplicadas (ignorando CustomerID): 556


## 3. Redundância de coluna: `loan_percent_income`

A Etapa 3 exige criar, na base de crédito, a coluna
`comprometimento_renda = (loan_amnt / person_income) * 100`. A base de crédito já
traz uma coluna `loan_percent_income` — verificamos se ela é, na prática, a mesma
informação (o que motivou remover a versão original dos preditores em vez de manter
as duas).

In [5]:
calculada = (credito['loan_amnt'] / credito['person_income']) * 100
existente = credito['loan_percent_income'] * 100  # loan_percent_income está em fração (0-1)

correlacao = existente.corr(calculada)
diferenca_absoluta = (existente - calculada).abs()

print('Correlação entre loan_percent_income*100 e comprometimento_renda calculado:', round(correlacao, 4))
print('Diferença absoluta média (pontos percentuais):', round(diferenca_absoluta.mean(), 3))
print('Linhas com diferença menor que 0,5 p.p.:', round((diferenca_absoluta < 0.5).mean() * 100, 1), '%')

Correlação entre loan_percent_income*100 e comprometimento_renda calculado: 0.9989
Diferença absoluta média (pontos percentuais): 0.276
Linhas com diferença menor que 0,5 p.p.: 96.7 %


A correlação acima de 0,99 confirma que `loan_percent_income` já é, para fins
práticos, o `comprometimento_renda` exigido pelo problema — a pequena diferença
vem de arredondamento na base original. Por isso a coluna original é descartada dos
preditores na Etapa 4 do pipeline principal, evitando manter duas colunas
redundantes.

## 4. Tabela-resumo consolidada

A mesma tabela publicada no `README.md`, agora calculada célula por célula a
partir dos dois arquivos CSV originais — não digitada à mão.

In [6]:
resumo = pd.DataFrame(
    {
        'Crédito': [
            f"{credito.shape[0]:,}".replace(',', '.'),
            credito.shape[1],
            f"{alvo_credito.loc[1]:.2f}% inadimplentes",
            len(nulos_credito),
            dup_credito,
        ],
        'E-commerce': [
            f"{ecommerce.shape[0]:,}".replace(',', '.'),
            ecommerce.shape[1],
            f"{alvo_ecommerce.loc[1]:.2f}% abandonos",
            len(nulos_ecommerce),
            dup_ecommerce_com_id,
        ],
    },
    index=['Registros', 'Colunas', 'Classe positiva', 'Colunas com nulos', 'Linhas duplicadas'],
)
resumo

,Crédito,E-commerce
Registros,32.581,5.630
Colunas,12,20
Classe positiva,21.82% inadimplentes,16.84% abandonos
Colunas com nulos,2,7
Linhas duplicadas,165,0


## Conclusão

Os números acima, calculados diretamente dos dois CSVs originais, sustentam a
escolha feita: a base de crédito tem quase 6x mais registros, problemas de
qualidade de dados reais e verificáveis (nulos em duas colunas, 165 duplicatas
exatas) que justificam decisões de tratamento fundamentadas nas Etapas 1 e 2, e uma
redundância de coluna (`loan_percent_income`) que conecta diretamente com a
exigência da Etapa 3. Além disso, o problema de negócio — decidir se um cliente
recebe ou não um empréstimo — é uma decisão binária com custo assimétrico direto
entre Falso Positivo e Falso Negativo, o que dá material rico para a análise de
impacto financeiro pedida na Etapa 6.

Por esses motivos, a base de crédito (Opção A) foi a escolhida para o Desafio 2.